# SGLang 原生 API

除了 OpenAI 兼容 API 之外，SGLang Runtime 还提供了原生服务器 API。我们介绍以下 API：

- `/generate`（文本生成模型）
- `/get_model_info`
- `/get_server_info`
- `/health`
- `/health_generate`
- `/flush_cache`
- `/update_weights`
- `/encode`（嵌入模型）
- `/v1/rerank`（交叉编码器重排序模型）
- `/v1/score`（仅解码器评分）
- `/classify`（奖励模型）
- `/start_expert_distribution_record`
- `/stop_expert_distribution_record`
- `/dump_expert_distribution_record`
- `/tokenize`
- `/detokenize`
- 完整的 API 列表请参见 [http_server.py](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/entrypoints/http_server.py)

在以下示例中，我们主要使用 `requests` 来测试这些 API。您也可以使用 `curl`。


## 启动服务器

In [ ]:
from sglang.test.doc_patch import launch_server_cmd
from sglang.utils import wait_for_server, print_highlight, terminate_process

server_process, port = launch_server_cmd(
    "python3 -m sglang.launch_server --model-path qwen/qwen2.5-0.5b-instruct --host 0.0.0.0 --log-level warning"
)

wait_for_server(f"http://localhost:{port}", process=server_process)

## Generate（文本生成模型）
生成补全。这与 OpenAI API 中的 `/v1/completions` 类似。详细参数请参见[采样参数](sampling_params.md)。

In [ ]:
import requests

url = f"http://localhost:{port}/generate"
data = {"text": "What is the capital of France?"}

response = requests.post(url, json=data)
print_highlight(response.json())

## 获取模型信息

获取模型的信息。

- `model_path`：模型的路径/名称。
- `is_generation`：模型是否用作生成模型或嵌入模型。
- `tokenizer_path`：tokenizer 的路径/名称。
- `preferred_sampling_params`：通过 `--preferred-sampling-params` 指定的默认采样参数。本示例中返回 `None`，因为我们未在服务器参数中显式配置。
- `weight_version`：此字段包含模型权重的版本。通常用于跟踪模型训练参数的变更或更新。
- `has_image_understanding`：模型是否具有图像理解能力。
- `has_audio_understanding`：模型是否具有音频理解能力。
- `model_type`：来自 HuggingFace 配置的模型类型（例如 "qwen2"、"llama"）。
- `architectures`：来自 HuggingFace 配置的模型架构（例如 ["Qwen2ForCausalLM"]）。

In [ ]:
url = f"http://localhost:{port}/get_model_info"

response = requests.get(url)
response_json = response.json()
print_highlight(response_json)
assert response_json["model_path"] == "qwen/qwen2.5-0.5b-instruct"
assert response_json["is_generation"] is True
assert response_json["tokenizer_path"] == "qwen/qwen2.5-0.5b-instruct"
assert response_json["preferred_sampling_params"] is None
assert response_json.keys() == {
    "model_path",
    "is_generation",
    "tokenizer_path",
    "preferred_sampling_params",
    "weight_version",
    "has_image_understanding",
    "has_audio_understanding",
    "model_type",
    "architectures",
}

## 获取服务器信息
获取服务器信息，包括 CLI 参数、token 限制和内存池大小。
- 注意：`get_server_info` 合并了以下已弃用的端点：
  - `get_server_args`
  - `get_memory_pool_size`
  - `get_max_total_num_tokens`

In [ ]:
url = f"http://localhost:{port}/get_server_info"

response = requests.get(url)
print_highlight(response.text)

## 健康检查
- `/health`：检查服务器健康状态。
- `/health_generate`：通过生成一个 token 来检查服务器健康状态。

In [ ]:
url = f"http://localhost:{port}/health_generate"

response = requests.get(url)
print_highlight(response.text)

In [ ]:
url = f"http://localhost:{port}/health"

response = requests.get(url)
print_highlight(response.text)

## 刷新缓存

刷新 radix 缓存。当通过 `/update_weights` API 更新模型权重时，此操作会自动触发。

In [ ]:
url = f"http://localhost:{port}/flush_cache"

response = requests.post(url)
print_highlight(response.text)

## 从磁盘更新权重

在不重启服务器的情况下从磁盘更新模型权重。仅适用于相同架构和参数大小的模型。

SGLang 支持 `update_weights_from_disk` API，用于训练期间的持续评估（将检查点保存到磁盘并从磁盘更新权重）。


In [ ]:
# successful update with same architecture and size

url = f"http://localhost:{port}/update_weights_from_disk"
data = {"model_path": "qwen/qwen2.5-0.5b-instruct"}

response = requests.post(url, json=data)
print_highlight(response.text)
assert response.json()["success"] is True
assert response.json()["message"] == "Succeeded to update model weights."

In [ ]:
# failed update with different parameter size or wrong name

url = f"http://localhost:{port}/update_weights_from_disk"
data = {"model_path": "qwen/qwen2.5-0.5b-instruct-wrong"}

response = requests.post(url, json=data)
response_json = response.json()
print_highlight(response_json)
assert response_json["success"] is False
assert response_json["message"] == (
    "Failed to get weights iterator: "
    "qwen/qwen2.5-0.5b-instruct-wrong"
    " (repository not found)."
)

In [ ]:
terminate_process(server_process)

## Encode（嵌入模型）

将文本编码为嵌入向量。请注意，此 API 仅适用于[嵌入模型](openai_api_embeddings.ipynb)，对生成模型会引发错误。
因此，我们启动一个新服务器来服务嵌入模型。

In [ ]:
embedding_process, port = launch_server_cmd("""
python3 -m sglang.launch_server --model-path Alibaba-NLP/gte-Qwen2-1.5B-instruct \
    --host 0.0.0.0 --is-embedding --log-level warning
""")

wait_for_server(f"http://localhost:{port}", process=embedding_process)

In [ ]:
# successful encode for embedding model

url = f"http://localhost:{port}/encode"
data = {"model": "Alibaba-NLP/gte-Qwen2-1.5B-instruct", "text": "Once upon a time"}

response = requests.post(url, json=data)
response_json = response.json()
print_highlight(f"Text embedding (first 10): {response_json['embedding'][:10]}")

In [ ]:
terminate_process(embedding_process)

## v1/rerank（交叉编码器重排序模型）
使用交叉编码器模型根据查询对文档列表进行重排序。请注意，此 API 仅适用于交叉编码器模型（如 [BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)），`attention-backend` 需为 `triton` 或 `torch_native`。


In [ ]:
reranker_process, port = launch_server_cmd("""
python3 -m sglang.launch_server --model-path BAAI/bge-reranker-v2-m3 \
    --host 0.0.0.0 --disable-radix-cache --chunked-prefill-size -1 --attention-backend triton --is-embedding --log-level warning
""")

wait_for_server(f"http://localhost:{port}", process=reranker_process)

In [ ]:
# compute rerank scores for query and documents

url = f"http://localhost:{port}/v1/rerank"
data = {
    "model": "BAAI/bge-reranker-v2-m3",
    "query": "what is panda?",
    "documents": [
        "hi",
        "The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.",
    ],
}

response = requests.post(url, json=data)
response_json = response.json()
for item in response_json:
    print_highlight(f"Score: {item['score']:.2f} - Document: '{item['document']}'")

In [ ]:
terminate_process(reranker_process)

## v1/score（仅解码器评分）

计算给定查询和项目的指定 token 概率。这对分类任务、响应评分或计算对数概率非常有用。

参数：
- `query`：查询文本
- `items`：要评分的项目文本
- `label_token_ids`：要计算概率的 token ID
- `apply_softmax`：是否应用 softmax 获取归一化概率（默认：False）
- `item_first`：拼接顺序中项目是否在前（默认：False）
- `model`：模型名称

响应包含 `scores` —— 每个项目一个概率列表，每个列表按 `label_token_ids` 的顺序排列。

In [ ]:
score_process, port = launch_server_cmd("""
python3 -m sglang.launch_server --model-path qwen/qwen2.5-0.5b-instruct \
    --host 0.0.0.0 --log-level warning
""")

wait_for_server(f"http://localhost:{port}", process=score_process)

In [ ]:
# Score the probability of different completions given a query
query = "The capital of France is"
items = ["Paris", "London", "Berlin"]

url = f"http://localhost:{port}/v1/score"
data = {
    "model": "qwen/qwen2.5-0.5b-instruct",
    "query": query,
    "items": items,
    "label_token_ids": [9454, 2753],  # e.g. "Yes" and "No" token ids
    "apply_softmax": True,  # Normalize probabilities to sum to 1
}

response = requests.post(url, json=data)
response_json = response.json()

# Display scores for each item
for item, scores in zip(items, response_json["scores"]):
    print_highlight(f"Item '{item}': probabilities = {[f'{s:.4f}' for s in scores]}")

In [ ]:
terminate_process(score_process)

## Classify（奖励模型）

SGLang Runtime 也支持奖励模型。这里我们使用奖励模型来分类成对生成结果的质量。

In [ ]:
# Note that SGLang now treats embedding models and reward models as the same type of models.
# This will be updated in the future.

reward_process, port = launch_server_cmd("""
python3 -m sglang.launch_server --model-path Skywork/Skywork-Reward-Llama-3.1-8B-v0.2 --host 0.0.0.0 --is-embedding --log-level warning
""")

wait_for_server(f"http://localhost:{port}", process=reward_process)

In [ ]:
from transformers import AutoTokenizer

PROMPT = (
    "What is the range of the numeric output of a sigmoid node in a neural network?"
)

RESPONSE1 = "The output of a sigmoid node is bounded between -1 and 1."
RESPONSE2 = "The output of a sigmoid node is bounded between 0 and 1."

CONVS = [
    [{"role": "user", "content": PROMPT}, {"role": "assistant", "content": RESPONSE1}],
    [{"role": "user", "content": PROMPT}, {"role": "assistant", "content": RESPONSE2}],
]

tokenizer = AutoTokenizer.from_pretrained("Skywork/Skywork-Reward-Llama-3.1-8B-v0.2")
prompts = tokenizer.apply_chat_template(CONVS, tokenize=False, return_dict=False)

url = f"http://localhost:{port}/classify"
data = {"model": "Skywork/Skywork-Reward-Llama-3.1-8B-v0.2", "text": prompts}

responses = requests.post(url, json=data).json()
for response in responses:
    print_highlight(f"reward: {response['embedding'][0]}")

In [ ]:
terminate_process(reward_process)

## 捕获 MoE 模型中的专家选择分布

SGLang Runtime 支持记录 MoE 模型运行中每个专家被选择的次数。这对于分析模型吞吐量和规划优化非常有用。

*注意：为了更好的可读性，我们仅打印 csv 的前 10 行。如果您想更深入地分析结果，请相应调整。*

In [ ]:
expert_record_server_process, port = launch_server_cmd(
    "python3 -m sglang.launch_server --model-path Qwen/Qwen1.5-MoE-A2.7B --host 0.0.0.0 --expert-distribution-recorder-mode stat --log-level warning"
)

wait_for_server(f"http://localhost:{port}", process=expert_record_server_process)

In [ ]:
response = requests.post(f"http://localhost:{port}/start_expert_distribution_record")
print_highlight(response)

url = f"http://localhost:{port}/generate"
data = {"text": "What is the capital of France?"}

response = requests.post(url, json=data)
print_highlight(response.json())

response = requests.post(f"http://localhost:{port}/stop_expert_distribution_record")
print_highlight(response)

response = requests.post(f"http://localhost:{port}/dump_expert_distribution_record")
print_highlight(response)

In [ ]:
terminate_process(expert_record_server_process)

## Tokenize/Detokenize 示例（往返测试）

本示例演示如何一起使用 /tokenize 和 /detokenize 端点。我们首先对字符串进行 tokenize，然后对生成的 ID 进行 detokenize 以重建原始文本。当您需要在外部处理 tokenization 但仍希望利用服务器进行 detokenization 时，此工作流非常有用。

In [ ]:
tokenizer_free_server_process, port = launch_server_cmd("""
python3 -m sglang.launch_server --model-path qwen/qwen2.5-0.5b-instruct
""")

wait_for_server(f"http://localhost:{port}", process=tokenizer_free_server_process)

In [ ]:
import requests
from sglang.utils import print_highlight

base_url = f"http://localhost:{port}"
tokenize_url = f"{base_url}/tokenize"
detokenize_url = f"{base_url}/detokenize"

model_name = "qwen/qwen2.5-0.5b-instruct"
input_text = "SGLang provides efficient tokenization endpoints."
print_highlight(f"Original Input Text:\n'{input_text}'")

# --- tokenize the input text ---
tokenize_payload = {
    "model": model_name,
    "prompt": input_text,
    "add_special_tokens": False,
}
try:
    tokenize_response = requests.post(tokenize_url, json=tokenize_payload)
    tokenize_response.raise_for_status()
    tokenization_result = tokenize_response.json()
    token_ids = tokenization_result.get("tokens")

    if not token_ids:
        raise ValueError("Tokenization returned empty tokens.")

    print_highlight(f"\nTokenized Output (IDs):\n{token_ids}")
    print_highlight(f"Token Count: {tokenization_result.get('count')}")
    print_highlight(f"Max Model Length: {tokenization_result.get('max_model_len')}")

    # --- detokenize the obtained token IDs ---
    detokenize_payload = {
        "model": model_name,
        "tokens": token_ids,
        "skip_special_tokens": True,
    }

    detokenize_response = requests.post(detokenize_url, json=detokenize_payload)
    detokenize_response.raise_for_status()
    detokenization_result = detokenize_response.json()
    reconstructed_text = detokenization_result.get("text")

    print_highlight(f"\nDetokenized Output (Text):\n'{reconstructed_text}'")

    if input_text == reconstructed_text:
        print_highlight(
            "\nRound Trip Successful: Original and reconstructed text match."
        )
    else:
        print_highlight(
            "\nRound Trip Mismatch: Original and reconstructed text differ."
        )

except requests.exceptions.RequestException as e:
    print_highlight(f"\nHTTP Request Error: {e}")
except Exception as e:
    print_highlight(f"\nAn error occurred: {e}")

In [ ]:
terminate_process(tokenizer_free_server_process)